In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA06 - Donations
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit, either directly or through the use of 
   third parties, make any domestic/international charitable donations?
   
   LOGIC SUMMARY:
   1. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is entirely excluded 
      based on the business logic.
   2. CATEGORY CODE FILTER: Strictly filters the Finance data for Category Codes 
      '292' and '427' (Charitable Donations).
   3. MAPPING & OUTPUT: Joins the standardized Cost Center Mapping Bootstrap to the 
      flagged transactions using safe Integer casting. Rolls up by AU_ID. If any 
      donation transaction exists for the Assessable Unit, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Combined_Finance_Raw AS ( 
    -- 1. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Donation_Transactions AS (
    -- 2. Clean the Finance columns and filter STRICTLY for donation category codes
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN ('292', '427')
),

Mapped_AUs AS (
    -- 3a. Join the flagged transactions to the standardized CC Mapping
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        CASE WHEN d.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END AS Has_Donation
    FROM vw_cost_center_mapping_bootstrap m
    
    -- LEFT JOIN so all AUs appear in the final output
    LEFT JOIN Donation_Transactions d
        ON CAST(m.Cost_Center_ID AS INT) = d.Cleaned_CC
)

-- 3b. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EBA06' AS QuestionID,               
    'Applicable' AS Applicability,       
    '' AS ApplicabilityRationale,        
    
    CASE 
        WHEN SUM(Has_Donation) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA06 - Donations Drill-Down
   
   PURPOSE: 
   Provides a row-by-row view of every Cost Center transaction that triggered a 'Yes' 
   for EBA06. Verifies that the matched Category Code is exactly 292 or 427.
=================================================================================== */

WITH Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Donation_Transactions AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        CostCenter AS Raw_Cost_Center_String
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN ('292', '427')
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    d.Raw_Cost_Center_String,
    d.CatCode AS Extracted_Category_Code,
    CASE 
        WHEN d.CatCode = '292' THEN 'Donation Code 292'
        WHEN d.CatCode = '427' THEN 'Donation Code 427'
        ELSE 'Unknown'
    END AS Category_Description
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN to only return AUs that successfully mapped to a donation transaction
INNER JOIN Donation_Transactions d
    ON CAST(m.Cost_Center_ID AS INT) = d.Cleaned_CC
    
ORDER BY BusinessID;